# PyTorch Tensor Basics

A targeted recap of tensor mechanics, scoped toward what attention/transformer implementation actually needs: indexing/slicing, memory layout (`view` vs `reshape` vs `.contiguous()`), `transpose`/`permute`, broadcasting, `matmul`, and `softmax`. Not a full generic tensor tutorial -- picking up specific rust, not starting from zero.

## Indexing and slicing

Same slicing syntax as Python lists/NumPy, extended to N dimensions — one slice/index per axis, separated by commas: `t[dim0_slice, dim1_slice, ...]`.

- `t[i]` — index along the first axis (returns a tensor with one fewer dimension).
- `t[i, j]` — index along the first two axes.
- `t[a:b]` — slice along the first axis, `[a, b)`, like a Python list slice.
- `t[a:b, c:d]` — independent slice per axis.
- `t[..., 0]` — `...` (`Ellipsis`) means "all the axes I didn't mention" — fills in as many `:` as needed. Common for grabbing a fixed index on the *last* axis regardless of how many dimensions come before it (e.g. `t[..., 0]` on any-rank tensor grabs index 0 of the last dim).
- `t[-1]` — negative indices count from the end, same as Python.
- `t[::2]` — step slicing works too.
- **Slicing returns a *view***, not a copy — it shares the same underlying memory as the original tensor. Mutating the slice mutates the original.

In [ ]:
# TODO: warm up on indexing/slicing.
# t is shape (2, 3, 4) -- think of it as (batch, seq_len, d_model).
import torch
t = torch.arange(24).reshape(2, 3, 4)
print(t)

# 1. Get the first item in the batch -> shape should be (3, 4)

# 2. Get the last item along the last dimension, for every batch/seq position -> shape (2, 3)
#    (use Ellipsis)

# 3. Get every other position along the seq_len dimension, for the whole batch

# 4. Take a slice of t, mutate an element in the slice, then print the original t --
#    confirm it changed too (proving slicing returns a view, not a copy)

## Group 1: Creation & inspection

`torch.tensor()`, `torch.arange()`, `torch.zeros()`/`torch.ones()`/`torch.randn()` (+ `*_like` variants), `torch.eye()`, `.shape`/`.size()`, `.dtype`, `.device`, `.numel()`

In [ ]:
# Group 1: Creation & inspection
# torch.tensor(), torch.arange(), torch.zeros()/torch.ones()/torch.randn() (+ *_like),
# torch.eye(), .shape/.size(), .dtype, .device, .numel()

import torch

## Group 2: Reshaping & layout

`.view()`, `.reshape()`, `.transpose()`, `.permute()`, `.contiguous()`, `.squeeze()`/`.unsqueeze()`, `.flatten()`, `.expand()`, `.repeat_interleave()` (also how grouped/multi-query attention broadcasts fewer KV heads to match more Q heads)

In [ ]:
# Group 2: Reshaping & layout
# .view(), .reshape(), .transpose(), .permute(), .contiguous(), .squeeze()/.unsqueeze(),
# .flatten(), .expand(), .repeat_interleave()


## Group 3: Copying vs. views

`.clone()` — contrast with slicing/`.view()`, which share memory with the original.

In [ ]:
# Group 3: Copying vs. views
# .clone() -- contrast with slicing/.view(), which share memory with the original


## Group 4: Combining & splitting

`torch.cat()` (also how a KV cache appends new keys/values along the sequence dim each generation step), `torch.stack()`, `.split()`, `.chunk()`

In [ ]:
# Group 4: Combining & splitting
# torch.cat(), torch.stack(), .split(), .chunk()


## Group 5: Core math

`@`/`torch.matmul()`, `torch.bmm()`, `torch.einsum()`, `torch.softmax()`, `F.log_softmax()`, elementwise `+ - * /`, `torch.exp()`, `torch.log()`, `torch.sqrt()`, `torch.rsqrt()`, `torch.cos()`/`torch.sin()` (rotary embeddings), `torch.sum()`/`.mean()` (with `dim=`), `torch.max()`/`.argmax()`, `torch.topk()`

In [ ]:
# Group 5: Core math
# @ / torch.matmul(), torch.bmm(), torch.einsum(), torch.softmax(), F.log_softmax(),
# elementwise + - * /, torch.exp(), torch.log(), torch.sqrt(), torch.rsqrt(),
# torch.cos()/torch.sin() (rotary embeddings), torch.sum()/.mean() (dim=),
# torch.max()/.argmax(), torch.topk()


## Group 6: Masking & selecting

Boolean indexing (`t[mask]`), `torch.where()`, `.masked_fill()`, `torch.triu()`/`torch.tril()` (causal mask; a banded combination of the two also builds a sliding-window mask), `torch.gather()`

In [ ]:
# Group 6: Masking & selecting
# boolean indexing (t[mask]), torch.where(), .masked_fill(), torch.triu()/torch.tril()
# (causal mask; banded triu+tril also builds a sliding-window mask), torch.gather()


## Group 7: LLM-specific layers/ops

`nn.Embedding`, `F.layer_norm()`/`nn.LayerNorm`, `F.gelu()`, `F.cross_entropy()`, `F.kl_div()` (policy-vs-reference penalty in PPO-style RLHF), `F.scaled_dot_product_attention()` (dispatches to a flash-attention kernel automatically when conditions allow), `F.pad()`

In [ ]:
# Group 7: LLM-specific layers/ops
# nn.Embedding, F.layer_norm()/nn.LayerNorm, F.gelu(), F.cross_entropy(),
# F.kl_div() (RLHF policy-vs-reference penalty),
# F.scaled_dot_product_attention() (auto-dispatches to flash attention), F.pad()

import torch.nn as nn
import torch.nn.functional as F

## Group 8: Sampling / generation

`torch.multinomial()`, `torch.topk()` (cross-ref Group 5), `.argmax()` for greedy decoding

In [ ]:
# Group 8: Sampling / generation
# torch.multinomial(), torch.topk() (cross-ref Group 5), .argmax() for greedy decoding


## Group 9: Device, dtype & inference mode

`.to(device)`/`.to(dtype)`, `.float()`/`.half()`, `.item()`, `torch.no_grad()`, `.detach()` (stop gradient flow into a frozen/reference or reward model — related to `no_grad()` but detaches a specific tensor rather than a whole block), `torch.clamp()`

In [ ]:
# Group 9: Device, dtype & inference mode
# .to(device)/.to(dtype), .float()/.half(), .item(), torch.no_grad(), .detach(),
# torch.clamp()


## Group 10: Debugging/testing

`torch.allclose()`, `torch.equal()`

In [ ]:
# Group 10: Debugging/testing
# torch.allclose(), torch.equal()
